# Examine mean-centering — baseline vs. per-model vs. pooled

Style and core diagnostic reused from the original
`compute_embeds_mean.ipynb` sample notebook: pairwise cosine
similarity between candidate embeddings, before and after
mean-centering, viewed as a histogram. That notebook centered one
trial's embeddings on their own mean and eyeballed the histogram
shift; this one asks a sharper, 3-way version of the same question
per model:

1. **Baseline** — raw embeddings, no centering.
2. **Per-model centering** — each model centered on *its own*
   fixed mean (`embeds_mean--<tag>--qwen-prm.npy`).
3. **Pooled centering** — each model centered on the *pooled* mean
   across all 5 models (`embeds_mean--pooled--qwen-prm.npy`).

If per-model and pooled centering produce visibly different pairwise-
similarity histograms, that's direct evidence per-model means are
doing something the pooled mean doesn't — complementing (not
replacing) the within/between variance-ratio diagnostic in
`examine_embeds_mean.ipynb`, which asked the same question from the
mean's own geometry rather than its downstream effect on similarity.

Depends on `results/embeds_mean/level-5/` already existing (produced
by `compute_embeds_mean.py`).


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

MEAN_DIR = "results/embeds_mean/level-5"

with open(f"{MEAN_DIR}/embeds_mean--manifest.json", encoding="utf-8") as fin:
    manifest = json.load(fin)

model_tags = list(manifest["models"].keys())
print(f"models: {model_tags}")
print(f"embeds_pipeline: {manifest['embeds_pipeline']}")


In [ ]:
# Load each model's raw pooled/projected embeddings, its own mean,
# and the pooled mean -- same manifest fields examine_embeds_mean.ipynb
# uses, so a .npy found later is never ambiguous about its source.
raw = {tag: np.load(manifest["models"][tag]["raw_npy"]) for tag in model_tags}
own_mean = {tag: np.load(manifest["models"][tag]["npy"]).flatten() for tag in model_tags}
pooled_mean = np.load(manifest["pooled"]["npy"]).flatten()

for tag in model_tags:
    print(f"{tag:16s} n={raw[tag].shape[0]:5d}  dim={raw[tag].shape[1]}")


## Pairwise cosine similarity — the reused diagnostic

Same computation as the sample notebook's active cells: full pairwise
cosine similarity (`X @ X.T` — the pooled/projected embeddings are
already L2-normalized by `_center_and_normalize` in the real search
pipeline, so a raw dot product IS cosine similarity here; centering
here is applied to the SAME pre-normalized vectors the sample
notebook worked with, matching its intent), then the upper triangle
via `np.triu_indices_from(..., k=1)` to drop the diagonal and each
pair's mirror.

One helper covers all three centering modes (`mean=None` reproduces
the sample notebook's uncentered baseline exactly; a mean array
reproduces its "subtract, then re-normalize before comparing" step —
made explicit here since the real pipeline always re-normalizes
after centering, see `_center_and_normalize`).


In [ ]:
def pairwise_cosine_sim(X, mean=None):
    """Center (optional) then L2-renormalize, matching
    core.mcts_sem_search_v02_00_00._center_and_normalize's order of
    operations, then return the upper-triangle pairwise cosine
    similarities (diagonal and mirrored pairs dropped)."""
    if mean is not None:
        X = X - mean
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = np.divide(X, norms, out=np.zeros_like(X), where=norms > 0)
    cosine_sim = X @ X.T
    iu = np.triu_indices_from(cosine_sim, k=1)
    return cosine_sim[iu]


## Per-model: 3-way histogram (baseline / per-model-centered / pooled-centered)

Same histogram styling as the sample notebook (`steelblue`, black
edges, dashed grid, 50 bins), overlaid per mode so the shift is
directly visible in one plot per model instead of three separate
ones.


In [ ]:
MODE_COLORS = {
    "baseline": "steelblue",
    "per-model centered": "darkorange",
    "pooled centered": "seagreen",
}

summary_rows = []

for tag in model_tags:
    X = raw[tag]
    sims = {
        "baseline": pairwise_cosine_sim(X, mean=None),
        "per-model centered": pairwise_cosine_sim(X, mean=own_mean[tag]),
        "pooled centered": pairwise_cosine_sim(X, mean=pooled_mean),
    }

    plt.figure(figsize=(9, 5))
    for mode, vals in sims.items():
        plt.hist(vals, bins=50, alpha=0.5, color=MODE_COLORS[mode],
                  edgecolor='black', linewidth=0.4, label=mode)
    plt.title(f'Pairwise Cosine Similarity — {tag}', fontsize=14)
    plt.xlabel('Cosine Similarity', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    for mode, vals in sims.items():
        summary_rows.append({
            "model": tag, "mode": mode,
            "mean_cos_sim": float(np.mean(vals)),
            "std_cos_sim": float(np.std(vals)),
        })


### Numerical-precision sanity check

Same check the sample notebook left as its last cell
(`unique_sims[unique_sims > 1.0]`) — cosine similarity of a unit
vector with itself is exactly 1.0 by construction, so values in the
upper triangle (mirrored pairs and the diagonal excluded) strictly
above 1.0 would indicate a normalization bug, not a real similarity.
Should print nothing.


In [ ]:
for tag in model_tags:
    baseline_sims = pairwise_cosine_sim(raw[tag], mean=None)
    over_one = baseline_sims[baseline_sims > 1.0]
    if over_one.size:
        print(f"{tag}: {over_one.size} values > 1.0 -- {over_one[:5]}")
print("OK: no pairwise cosine similarity exceeds 1.0" if all(
    pairwise_cosine_sim(raw[t], mean=None).max() <= 1.0 for t in model_tags
) else "See above -- investigate before trusting the histograms")


## Cross-model summary — does centering mode actually matter?

Mean pairwise cosine similarity per (model, mode) — lower means more
decorrelated (centering "worked" more). The gap between the orange
(per-model) and green (pooled) bars, for each model, is the direct
answer to whether a model needs its own mean or the pooled one would
do just as well for the thing centering is actually meant to
accomplish.


In [ ]:
import numpy as np

modes = ["baseline", "per-model centered", "pooled centered"]
x = np.arange(len(model_tags))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
for i, mode in enumerate(modes):
    means = [
        next(r["mean_cos_sim"] for r in summary_rows
             if r["model"] == tag and r["mode"] == mode)
        for tag in model_tags
    ]
    ax.bar(x + (i - 1) * width, means, width, label=mode,
           color=MODE_COLORS[mode], edgecolor='black', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(model_tags, rotation=30, ha="right")
ax.set_ylabel("Mean pairwise cosine similarity")
ax.set_title("Mean pairwise cosine similarity by model and centering mode")
ax.axhline(0, color="black", linewidth=0.6)
ax.legend()
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"{'model':16s} {'baseline':>10s} {'per-model':>10s} {'pooled':>10s}  gap(per-model - pooled)")
for tag in model_tags:
    vals = {r["mode"]: r["mean_cos_sim"] for r in summary_rows if r["model"] == tag}
    gap = vals["per-model centered"] - vals["pooled centered"]
    print(f"{tag:16s} {vals['baseline']:10.4f} {vals['per-model centered']:10.4f} "
          f"{vals['pooled centered']:10.4f}  {gap:+.4f}")


## Reading this together

- **Baseline vs. either centered mode**, for every model, is the
  first thing to check: centering should visibly shift the histogram
  left (lower mean pairwise similarity) — if it doesn't, mean-
  centering isn't doing its job for that model, independent of which
  mean is used.
- **Per-model vs. pooled centered, per model:** a small gap (the last
  column above, near 0) says the pooled mean decorrelates that
  model's candidates just as well as its own mean would — direct,
  downstream-effect confirmation of a small variance-ratio finding in
  `examine_embeds_mean.ipynb`. A large gap says that model's own mean
  is doing meaningfully more work than the pooled one, and
  `search.embeds_mean_dir` should point at the per-model `.npy` for
  that model's sem-mcts runs, not the pooled one.
- These two notebooks ask the same underlying question from two
  angles — mean geometry (`examine_embeds_mean.ipynb`) vs. downstream
  similarity effect (this one) — and should agree; if they don't,
  that disagreement itself is worth a closer look before picking a
  default `embeds_mean_dir`.
